In [ ]:
import os
import json
import time
from dataclasses import dataclass, asdict

import pandas as pd
import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

In [ ]:
# ============================================================
# CONFIG
# ============================================================

@dataclass
class CFG:
    # --- Experimento ---
    experiment_code: str = "B2"
    version: int = 1
    fold: int = 4
    k: int = 5
    experiment_name: str = f'experimento-{experiment_code}-overfitting'

    # --- Rutas ---
    dataset_path: str = "./dataset"
    # CSV generados por tu preparing_dataset (con columna "text")
    dataset_prefix: str = f"{experiment_code}-03-preparing-dataset-pt"  # sin -{fold}..., ajusta abajo
    # Modelo descargado con: hf download ... --local-dir ./models/llama3.1-8b
    base_model_path: str = "./models/llama3.1-8b"

    # --- Secuencia ---
    max_seq_len: int = 512

    # --- Entrenamiento ---
    seed: int = 42
    num_train_epochs: int = 10
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    lr_scheduler_type: str = "cosine"

    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8

    eval_steps: int = 200
    save_steps: int = 200
    logging_steps: int = 50
    save_total_limit: int = 2

    dataloader_num_workers: int = 4

    # --- QLoRA/LoRA ---
    use_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # --- TRL ---
    packing: bool = False


cfg = CFG()

# Offline (por si acaso)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:

# ============================================================
# HELPERS
# ============================================================

def make_output_dir(cfg: CFG) -> str:
    basename = f"04-{cfg.experiment_code}-v{cfg.version}-f{cfg.fold}-k{cfg.k}-{cfg.experiment_name}".replace(" ", "_")
    outdir = f"{basename}-outputs"
    os.makedirs(outdir, exist_ok=True)
    return outdir


def guess_train_val_paths(cfg: CFG):
    """
    Ajusta esto a cómo te genera los splits tu función stratified_kfold_cross_validation.
    En tu ejemplo anterior, tú usabas algo como:
      {output_file[:-4]} + '-'  y luego añadía sufijos.
    Tú antes entrenabas con:
      '{}/{}-train.csv'.format(dataset_path, dataset_basename)
    y dataset_basename = '{code}-03-preparing-dataset-pt-{fold}'
    """
    dataset_basename = f"{cfg.experiment_code}-03-preparing-dataset-pt-{cfg.fold}"
    train_path = f"{cfg.dataset_path}/{dataset_basename}-train.csv"
    val_path   = f"{cfg.dataset_path}/{dataset_basename}-val.csv"
    return train_path, val_path


def load_prepared_csv_as_dataset(train_path: str, val_path: str):
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)

    if "text" not in df_train.columns or "text" not in df_val.columns:
        raise ValueError(
            "Tus CSV no tienen columna 'text'.\n"
            "Asegúrate de haber añadido la columna 'text' en el preparing_dataset.\n"
            "Si no quieres columna 'text', dímelo y te adapto el training a source_text/target_text."
        )

    df_train["text"] = df_train["text"].astype(str)
    df_val["text"] = df_val["text"].astype(str)

    # Solo conservamos "text" (más limpio)
    train_ds = Dataset.from_pandas(df_train[["text"]])
    val_ds = Dataset.from_pandas(df_val[["text"]])
    return train_ds, val_ds


def save_run_config(outdir: str, cfg: CFG):
    with open(os.path.join(outdir, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)


In [ ]:
# ============================================================
# MAIN
# ============================================================

def main():
    set_seed(cfg.seed)

    outdir = make_output_dir(cfg)
    save_run_config(outdir, cfg)

    train_path, val_path = guess_train_val_paths(cfg)
    print("train_path:", train_path)
    print("val_path  :", val_path)
    print("outdir    :", outdir)

    if not os.path.exists(cfg.base_model_path):
        raise FileNotFoundError(
            f"No encuentro el modelo local en: {cfg.base_model_path}\n"
            "Descárgalo con:\n"
            "  hf download meta-llama/Meta-Llama-3.1-8B-Instruct --local-dir ./models/llama3.1-8b"
        )

    # --------------------------
    # DATA
    # --------------------------
    train_ds, val_ds = load_prepared_csv_as_dataset(train_path, val_path)
    print("Train examples:", len(train_ds), " Val examples:", len(val_ds))
    print("Sample text:\n", train_ds[0]["text"][:400], "...\n")

    # --------------------------
    # TOKENIZER
    # --------------------------
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model_path, use_fast=True, local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --------------------------
    # MODEL
    # --------------------------
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    if cfg.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            cfg.base_model_path,
            quantization_config=bnb_config,
            device_map="auto",
            local_files_only=True,
            torch_dtype=compute_dtype,
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.base_model_path,
            device_map="auto",
            local_files_only=True,
            torch_dtype=compute_dtype,
        )

    # LoRA
    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # --------------------------
    # TRAINING ARGS
    # --------------------------
    training_args = TrainingArguments(
        output_dir=outdir,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        per_device_eval_batch_size=cfg.per_device_eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        num_train_epochs=cfg.num_train_epochs,
        learning_rate=cfg.learning_rate,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,

        evaluation_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_steps=cfg.save_steps,
        logging_steps=cfg.logging_steps,
        save_total_limit=cfg.save_total_limit,

        report_to="none",
        dataloader_num_workers=cfg.dataloader_num_workers,

        bf16=torch.cuda.is_available(),  # si falla en tu GPU, cambia a fp16=True y bf16=False
        fp16=False,

        optim="paged_adamw_8bit" if cfg.use_4bit else "adamw_torch",
        remove_unused_columns=False,
    )

    # --------------------------
    # TRAINER
    # --------------------------
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        dataset_text_field="text",     # <- aquí está la clave: ya tienes "text" preparado
        max_seq_length=cfg.max_seq_len,
        args=training_args,
        packing=cfg.packing,
    )

    # --------------------------
    # TRAIN
    # --------------------------
    t0 = time.time()
    trainer.train()
    t1 = time.time()
    print(f"Training finished in {(t1 - t0)/60:.1f} min")

    # --------------------------
    # SAVE
    # --------------------------
    trainer.model.save_pretrained(outdir)   # guarda adaptadores LoRA
    tokenizer.save_pretrained(outdir)

    print("Saved to:", outdir)
    print("Listo.")


if __name__ == "__main__":
    main()